Load Data & Packages

In [8]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline

PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR.parent / "yelp_clean.csv"
MODEL_DIR = PART_B_DIR / "models"

In [9]:
# --- Same load and split as Q2, so the tuned model is scored on the identical test set ---
df = pd.read_csv(DATA_FILE, usecols=["clean_text", "sentiment"])

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

Train: 23998   Test: 6000


Hyperparameter Search

In [ ]:
# --- Two groups of hyperparameters: how the text is represented, and how the forest is grown ---
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

param_dist = {
    "tfidf__max_features": [10000, 20000, None],   # vocabulary cap
    "tfidf__ngram_range": [(1, 1), (1, 2)],        # bigrams capture phrases like "not good"
    "tfidf__min_df": [1, 2, 3],                    # discard rare / typo terms
    "tfidf__sublinear_tf": [True, False],          # log-scale term frequency
    "clf__n_estimators": [100, 200, 300],          # number of trees averaged
    "clf__max_depth": [None, 30],                  # tree size limit
    "clf__min_samples_leaf": [1, 2],               # leaf smoothing
    "clf__class_weight": [None, "balanced", "balanced_subsample"],  # 2:1:2 class imbalance
}

search = RandomizedSearchCV(
    pipeline, param_dist,
    n_iter=10, cv=3, scoring="f1_macro",
    random_state=42, n_jobs=1, verbose=1,
)
search.fit(X_train, y_train)
print("Search complete.")

Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [ ]:
# --- Winning configuration, and whether it actually beat the untuned Q2 baseline ---
print("=== Q3: Best Hyperparameters ===")
for param, value in sorted(search.best_params_.items()):
    name = param.replace("clf__", "").replace("tfidf__", "tfidf_")
    print(f"  {name:<24}: {value}")
print(f"Best CV Macro F1: {search.best_score_:.4f}")
print()

tuned = search.best_estimator_
joblib.dump(tuned, MODEL_DIR / "rf_tuned.joblib")
scores = {}
print(f"{'Model':<24}{'Accuracy':>10}{'Macro F1':>10}")
for name, model in [("Q2 baseline (defaults)", joblib.load(MODEL_DIR / "rf_pipeline.joblib")),
                    ("Q3 tuned", tuned)]:
    p = model.predict(X_test)
    scores[name] = (accuracy_score(y_test, p), f1_score(y_test, p, average="macro"))
    print(f"{name:<24}{scores[name][0]:>10.4f}{scores[name][1]:>10.4f}")

gain_acc = scores["Q3 tuned"][0] - scores["Q2 baseline (defaults)"][0]
gain_f1 = scores["Q3 tuned"][1] - scores["Q2 baseline (defaults)"][1]
print(f"{'Improvement':<24}{gain_acc:>+10.4f}{gain_f1:>+10.4f}")